# TracIn para detección de anomalías: nuestras mejoras

Equipo 6 (Alan Rangel, Carlos Bruzual) · Aprendizaje no supervisado, INAOE

El método del artículo (Verdone et al., 2026) **no entrena una red para identificar anomalías: usa el proceso de
aprendizaje de una red para detectarlas**. Entrena un LSTM a predecir la serie y mide con TracIn cuánto le costó aprender
cada parte; lo que más le costó es lo anómalo. Propusimos tres mejoras y aquí se prueba cada una.

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todo*. No hay que subir nada. Con la CPU gratuita de Colab tarda unos
40 minutos; casi todo es la sección 2. Después de correrla, las secciones 3 a 8 sólo leen resultados.

**Contenido**

0. Preparación y código
1. El método original
2. Correr los experimentos
3. **Mejora 1:** umbral sin etiquetas
4. **Mejora 2:** reponderación
5. **Mejora 3:** proyección al objetivo
6. Las mejoras combinadas
7. Comparación con otros métodos de detección
8. Resultados completos y resumen

Los experimentos usan las mismas semillas que nuestras corridas, así que cada tabla trae al lado nuestra cifra para
comparar (puede diferir en el tercer decimal por la versión de PyTorch o del procesador). Las corridas completas usaron
más semillas; esas cifras también se muestran.

## 0. Preparación y código

Las celdas de esta sección son el código que usamos, copiado de nuestro repositorio. Las tres primeras están ocultas (doble
clic para verlas): el generador de series sintéticas, las proyecciones y umbrales, y el protocolo de evaluación.

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import roc_auc_score
try:
    from IPython.display import display
except ImportError:
    display = print
warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)
print("PyTorch", torch.__version__, "· hilos:", torch.get_num_threads())

In [ ]:
# @title Generador de series sintéticas (Lai et al., 2021)
from __future__ import annotations
"""Generador de series sintéticas multivariadas con anomalías.

Copia fiel del generador oficial del benchmark de Lai et al. (2021),
``tods/benchmark/synthetic/Generator/multivariate_generator.py`` (rama
``benchmark`` de github.com/datamllab/tods), que es el que usa el artículo.
Sólo se cambió: (1) configuraciones copiadas por dataset para evitar efectos
colaterales entre llamadas y (2) una función ``make_dataset`` que arma los
15 datasets del artículo con los parámetros del ``__main__`` original.

Tipos de anomalía (Lai et al., 2021; Fig. 1 del artículo):
    0 = puntual global, 1 = puntual contextual, 2 = shapelet (colectiva global),
    3 = estacional, 4 = de tendencia.
El tipo t se inyecta en el canal t, como en el ``__main__`` original.
"""

import copy

import numpy as np


def sine(length, freq=0.04, coef=1.5, offset=0.0, noise_amp=0.05):
    timestamp = np.arange(length)
    value = np.sin(2 * np.pi * freq * timestamp)
    if noise_amp != 0:
        value = value + noise_amp * np.random.normal(0, 1, length)
    return coef * value + offset


def cosine(length, freq=0.04, coef=1.5, offset=0.0, noise_amp=0.05):
    timestamp = np.arange(length)
    value = np.cos(2 * np.pi * freq * timestamp)
    if noise_amp != 0:
        value = value + noise_amp * np.random.normal(0, 1, length)
    return coef * value + offset


def square_sine(level=5, length=500, freq=0.04, coef=1.5, offset=0.0, noise_amp=0.05):
    value = np.zeros(length)
    for i in range(level):
        value += 1 / (2 * i + 1) * sine(length=length, freq=freq * (2 * i + 1), coef=coef,
                                        offset=offset, noise_amp=noise_amp)
    return value


class MultivariateDataGenerator:
    def __init__(self, dim, stream_length, behavior, behavior_config):
        self.dim = dim
        self.STREAM_LENGTH = stream_length
        self.behavior = behavior
        self.behavior_config = copy.deepcopy(behavior_config)
        self.data = np.empty(shape=[0, stream_length], dtype=float)
        self.generate_timeseries()

    def generate_timeseries(self):
        for i in range(self.dim):
            self.behavior_config[i]["length"] = self.STREAM_LENGTH
            self.data = np.append(self.data, [self.behavior[i](**self.behavior_config[i])], axis=0)
        self.data_origin = self.data.copy()
        self.label = np.zeros(self.STREAM_LENGTH, dtype=int)

    def point_global_outliers(self, dim_no, ratio, factor, radius):
        position = (np.random.rand(round(self.STREAM_LENGTH * ratio)) * self.STREAM_LENGTH).astype(int)
        maximum, minimum = max(self.data[dim_no]), min(self.data[dim_no])
        for i in position:
            local_std = self.data_origin[dim_no][max(0, i - radius):min(i + radius, self.STREAM_LENGTH)].std()
            self.data[dim_no][i] = self.data_origin[dim_no][i] * factor * local_std
            if 0 <= self.data[dim_no][i] < maximum:
                self.data[dim_no][i] = maximum
            if 0 > self.data[dim_no][i] > minimum:
                self.data[dim_no][i] = minimum
            self.label[i] = 1

    def point_contextual_outliers(self, dim_no, ratio, factor, radius):
        position = (np.random.rand(round(self.STREAM_LENGTH * ratio)) * self.STREAM_LENGTH).astype(int)
        maximum, minimum = max(self.data[dim_no]), min(self.data[dim_no])
        for i in position:
            local_std = self.data_origin[dim_no][max(0, i - radius):min(i + radius, self.STREAM_LENGTH)].std()
            self.data[dim_no][i] = self.data_origin[dim_no][i] * factor * local_std
            if self.data[dim_no][i] > maximum:
                self.data[dim_no][i] = maximum * min(0.95, abs(np.random.normal(0, 1)))
            if self.data[dim_no][i] < minimum:
                self.data[dim_no][i] = minimum * min(0.95, abs(np.random.normal(0, 1)))
            self.label[i] = 1

    def collective_global_outliers(self, dim_no, ratio, radius, coef=3.0, noise_amp=0.0,
                                   level=5, freq=0.04, offset=0.0):
        position = (np.random.rand(round(self.STREAM_LENGTH * ratio / (2 * radius))) * self.STREAM_LENGTH).astype(int)
        sub_data = square_sine(level=level, length=self.STREAM_LENGTH, freq=freq,
                               coef=coef, offset=offset, noise_amp=noise_amp)
        for i in position:
            start, end = max(0, i - radius), min(self.STREAM_LENGTH, i + radius)
            self.data[dim_no][start:end] = sub_data[start:end]
            self.label[start:end] = 1

    def collective_trend_outliers(self, dim_no, ratio, factor, radius):
        position = (np.random.rand(round(self.STREAM_LENGTH * ratio / (2 * radius))) * self.STREAM_LENGTH).astype(int)
        for i in position:
            start, end = max(0, i - radius), min(self.STREAM_LENGTH, i + radius)
            slope = np.random.choice([-1, 1]) * factor * np.arange(end - start)
            self.data[dim_no][start:end] = self.data_origin[dim_no][start:end] + slope
            self.data[dim_no][end:] = self.data[dim_no][end:] + slope[-1]
            self.label[start:end] = 1

    def collective_seasonal_outliers(self, dim_no, ratio, factor, radius):
        position = (np.random.rand(round(self.STREAM_LENGTH * ratio / (2 * radius))) * self.STREAM_LENGTH).astype(int)
        seasonal_config = self.behavior_config[dim_no]
        seasonal_config["freq"] = factor * self.behavior_config[dim_no]["freq"]
        for i in position:
            start, end = max(0, i - radius), min(self.STREAM_LENGTH, i + radius)
            self.data[dim_no][start:end] = self.behavior[dim_no](**seasonal_config)[start:end]
            self.label[start:end] = 1


BEHAVIOR = [sine, cosine, sine, cosine, sine]
BEHAVIOR_CONFIG = [
    {"freq": 0.04, "coef": 1.5, "offset": 0.0, "noise_amp": 0.05},
    {"freq": 0.04, "coef": 2.5, "offset": 0.0, "noise_amp": 0.05},
    {"freq": 0.04, "coef": 1.5, "offset": 0.0, "noise_amp": 0.05},
    {"freq": 0.04, "coef": 2.5, "offset": 2.0, "noise_amp": 0.05},
    {"freq": 0.04, "coef": 1.5, "offset": -2.0, "noise_amp": 0.05},
]

# Los 15 datasets del artículo (Tabla 2).
DATASETS = ["0", "1", "2", "3", "4", "01", "12", "23", "34",
            "012", "123", "234", "0123", "1234", "01234"]


def make_dataset(types: str, seed: int, length: int = 400, ratio: float = 0.05):
    """Genera la serie ``types`` (p. ej. "012") con la semilla ``seed``.
    ``ratio`` es la proporción por tipo (0.05 en el benchmark original)."""
    np.random.seed(seed)
    g = MultivariateDataGenerator(dim=5, stream_length=length, behavior=BEHAVIOR,
                                  behavior_config=BEHAVIOR_CONFIG)
    # Mismo orden de inyección que en el __main__ original.
    if "0" in types:
        g.point_global_outliers(dim_no=0, ratio=ratio, factor=3.5, radius=5)
    if "1" in types:
        g.point_contextual_outliers(dim_no=1, ratio=ratio, factor=2.5, radius=5)
    if "2" in types:
        g.collective_global_outliers(dim_no=2, ratio=ratio, radius=5, coef=1.5, noise_amp=0.03,
                                     level=20, freq=0.04, offset=0.0)
    if "3" in types:
        g.collective_seasonal_outliers(dim_no=3, ratio=ratio, factor=3, radius=5)
    if "4" in types:
        g.collective_trend_outliers(dim_no=4, ratio=ratio, factor=0.5, radius=5)
    return g.data.T.astype(np.float32), g.label.astype(np.int8)

In [ ]:
# @title Proyecciones, umbrales sin etiquetas (mejora 1), reponderación (mejora 2) y LOF
from __future__ import annotations
from scipy import stats
from sklearn.neighbors import LocalOutlierFactor

EDGE = 19


def project_sum(sample_scores: np.ndarray, span: int, n: int) -> np.ndarray:
    """Ec. (4) del artículo: cada instante suma el score de todas las
    ventanas que lo contienen. La ventana i cubre [i, i + span)."""
    s = np.asarray(sample_scores, dtype=np.float64)
    acc = np.zeros(n + 1)
    m = len(s)
    acc[:m] += s
    acc[span:span + m] -= s
    return np.cumsum(acc)[:n]


def project_targets_mean(sample_scores: np.ndarray, L: int, P: int, n: int) -> np.ndarray:
    """Proyección clásica de *prediction deviation*: el error de la ventana i
    se asigna sólo a los instantes que predice, [i + L, i + L + P), y se
    promedia sobre las predicciones que cubren cada instante."""
    s = np.asarray(sample_scores, dtype=np.float64)
    m = len(s)
    acc = np.zeros(n + 1)
    cnt = np.zeros(n + 1)
    acc[L:L + m] += s
    acc[L + P:L + P + m] -= s
    cnt[L:L + m] += 1
    cnt[L + P:L + P + m] -= 1
    acc, cnt = np.cumsum(acc)[:n], np.cumsum(cnt)[:n]
    out = np.zeros(n)
    nz = cnt > 0
    out[nz] = acc[nz] / cnt[nz]
    return out


def valid_mask(n: int, edge: int = EDGE) -> np.ndarray:
    m = np.zeros(n, dtype=bool)
    m[edge:n - edge] = True
    return m


def _log(s: np.ndarray) -> np.ndarray:
    s = np.asarray(s, dtype=np.float64)
    pos = s[s > 0]
    floor = pos.min() * 0.5 if pos.size else 1e-12
    return np.log(np.maximum(s, floor))


def thr_iqr(s: np.ndarray, k: float = 1.5) -> float:
    q1, q3 = np.percentile(s, [25, 75])
    return float(q3 + k * (q3 - q1))


def thr_mad(s: np.ndarray, k: float = 3.0) -> float:
    med = np.median(s)
    mad = 1.4826 * np.median(np.abs(s - med))
    return float(med + k * mad)


def thr_pot(s: np.ndarray, q: float = 1e-3, level: float = 0.98) -> float:
    """Peaks-Over-Threshold (Siffer et al., 2017, SPOT en modo batch).
    Se ajusta una Pareto generalizada a los excesos sobre el cuantil
    ``level`` y se devuelve el cuantil z_q tal que P(S > z_q) = q."""
    s = np.asarray(s, dtype=np.float64)
    n = len(s)
    t = np.quantile(s, level)
    exc = s[s > t] - t
    nt = len(exc)
    if nt < 5:
        return float(s.max())
    try:
        gamma, _, sigma = stats.genpareto.fit(exc, floc=0)
    except Exception:  # pragma: no cover - ajuste numérico fallido
        return float(s.max())
    r = q * n / nt
    if abs(gamma) < 1e-8:
        return float(t - sigma * np.log(r))
    return float(t + (sigma / gamma) * (r ** (-gamma) - 1.0))


def unsupervised_thresholds(s: np.ndarray) -> dict[str, float]:
    """Todos los umbrales sin etiquetas, con hiperparámetros fijados a priori
    (k = 1.5 para IQR, k = 3 para MAD, q = 1e-3 para POT). Los de log-escala
    se devuelven ya transformados de vuelta a la escala original."""
    ls = _log(s)
    return {
        "iqr": thr_iqr(s),
        "mad": thr_mad(s),
        "log_iqr": float(np.exp(thr_iqr(ls))),
        "log_mad": float(np.exp(thr_mad(ls))),
        "pot": thr_pot(s),
    }


def robust_z(v: np.ndarray) -> np.ndarray:
    med = np.median(v)
    mad = 1.4826 * np.median(np.abs(v - med))
    return (v - med) / max(mad, 1e-12)


def weights_from_influence(s: np.ndarray, mode: str, z0: float = 2.0, w_min: float = 0.05) -> np.ndarray:
    """Pesos por ventana a partir de su self-influence (escala log, z robusto).

    * ``soft``: w = 1 si z <= z0;  w = max(w_min, exp(-(z - z0))) si z > z0.
      Baja el peso gradualmente sin descartar ninguna ventana.
    * ``trim``: w = 0 si z > z0 (recorte duro al estilo ITLM, Shen &
      Sanghavi, 2019). Se incluye como contraste de la versión suave.
    z0 = 2 y w_min = 0.05 se fijan a priori (no se ajustan con etiquetas)."""
    z = robust_z(np.log(np.maximum(s, 1e-30)))
    if mode == "soft":
        return np.where(z <= z0, 1.0, np.maximum(w_min, np.exp(-(z - z0))))
    if mode == "trim":
        return np.where(z <= z0, 1.0, 0.0)
    raise ValueError(mode)


def adapt_weights(si: np.ndarray) -> np.ndarray:
    return np.where(si > thr_iqr(si), 0.0, 1.0)


def window_matrix(X: np.ndarray, w: int) -> np.ndarray:
    if w == 1:
        return X
    v = np.lib.stride_tricks.sliding_window_view(X, w, axis=0)  # (n-w+1, C, w)
    return np.ascontiguousarray(v.transpose(0, 2, 1).reshape(len(v), -1))


def _to_timesteps(win_scores: np.ndarray, w: int, n: int) -> np.ndarray:
    return project_sum(win_scores, w, n)


def lof(X, w, k=20, device="cpu"):
    Z = window_matrix(X, w)
    if device == "cpu" or len(Z) < 20000:
        m = LocalOutlierFactor(n_neighbors=k).fit(Z)
        return _to_timesteps(-m.negative_outlier_factor_, w, len(X))
    return _to_timesteps(lof_gpu(Z, k, device), w, len(X))

In [ ]:
# @title Protocolo de evaluación del artículo (umbral dinámico, F1 binario y macro)
from __future__ import annotations
from sklearn.metrics import average_precision_score

PCTS = np.arange(50.0, 99.95, 0.1)


def f1_pair(y: np.ndarray, p: np.ndarray) -> tuple[float, float]:
    """(F1 binario de la clase anómala, F1 macro)."""
    tp = np.sum(y & p); fp = np.sum(~y & p); fn = np.sum(y & ~p); tn = np.sum(~y & ~p)
    fa = 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0
    fn_ = 2 * tn / (2 * tn + fn + fp) if tn + fn + fp else 0.0
    return float(fa), float((fa + fn_) / 2)


def oracle_f1(y, s):
    order = np.argsort(-s, kind="stable")
    ys = y[order]
    tp = np.cumsum(ys)
    k = np.arange(1, len(ys) + 1)
    return float(np.max(2 * tp / (k + ys.sum()))) if ys.sum() else float("nan")


def _best_pct(y, s, s_all, crit):
    """Percentil (de la distribución completa del score) que maximiza el F1
    en (y, s). Vectorizado sobre la grilla de percentiles."""
    thr = np.percentile(s_all, PCTS)
    P = s[None, :] >= thr[:, None]
    Y = y[None, :]
    tp = (P & Y).sum(1); fp = (P & ~Y).sum(1); fn = (~P & Y).sum(1); tn = (~P & ~Y).sum(1)
    fa = np.where(tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
    fnm = np.where(tn + fn + fp > 0, 2 * tn / np.maximum(2 * tn + fn + fp, 1), 0.0)
    f = fa if crit == "bin" else (fa + fnm) / 2
    return thr[int(np.argmax(f))]


def evaluate(y: np.ndarray, s: np.ndarray, tau: float, prop: float = 0.5) -> dict:
    y = y.astype(bool)
    s = np.asarray(s, dtype=np.float64)
    s = s / s.max()
    out = {"auprc": float(average_precision_score(y, s)), "oracle": oracle_f1(y, s)}
    for nm, thr in (("pct", np.percentile(s, tau * 100)), ("frac", tau)):
        b, m = f1_pair(y, s >= thr)
        out[f"S_{nm}_bin"], out[f"S_{nm}_mac"] = b, m
    k = int(len(s) * prop)
    for crit in ("bin", "mac"):
        thr = _best_pct(y[:k], s[:k], s, crit)
        b, m = f1_pair(y[k:], s[k:] >= thr)
        out[f"D_{crit}"] = b if crit == "bin" else m
    return out

**Modelo y TracIn.** Un LSTM recibe 4 instantes de la serie y predice el siguiente. Al final de cada época se guarda, para
cada ventana, el cuadrado del gradiente de su pérdida por la tasa de aprendizaje; la suma sobre las épocas es la
*self-influence* de TracIn. Con la última capa y pérdida cuadrática tiene forma cerrada:

$$SI(x) = \sum_k \eta_k \, \frac{4}{D} \, \ell_k(x) \, (1 + \lVert h_k(x) \rVert^2)$$

es decir, el error de cada ventana acumulado durante todo el entrenamiento. TracIn no cambia el entrenamiento, sólo lo
observa. El parámetro `weights` de `run` es lo que usa la mejora 2.

In [ ]:
from __future__ import annotations
import math
from dataclasses import dataclass, asdict
from torch import nn
from torch.func import functional_call, grad, vmap

@dataclass(frozen=True)
class Cfg:
    lr: float = 1e-3
    batch: int = 32
    max_epochs: int = 100
    patience: int = 10
    hidden: int = 64
    sched: bool = True
    val_frac: float = 0.0
    norm: str = "z"
    grad: str = "last"
    lr_weight: bool = True

    def tag(self) -> str:
        d = asdict(self)
        base = asdict(Cfg())
        diff = {k: v for k, v in d.items() if v != base[k]}
        return "base" if not diff else ",".join(f"{k}={v}" for k, v in diff.items())


class LSTM(nn.Module):
    """LSTM de una capa + capa lineal final (como en ``forecaster.py``)."""

    def __init__(self, C: int, P: int, hidden: int):
        super().__init__()
        self.P, self.C, self.H = P, C, hidden
        self.ih = nn.Linear(C, 4 * hidden)
        self.hh = nn.Linear(hidden, 4 * hidden)
        self.head = nn.Linear(hidden, P * C)
        k = 1 / math.sqrt(hidden)
        for p in list(self.ih.parameters()) + list(self.hh.parameters()):
            nn.init.uniform_(p, -k, k)

    def forward(self, x):  # x: (B, L, C)
        B = x.shape[0]
        h = x.new_zeros(B, self.H)
        c = x.new_zeros(B, self.H)
        gx = self.ih(x)
        for t in range(x.shape[1]):
            i, f, g, o = (gx[:, t] + self.hh(h)).chunk(4, dim=1)
            c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
            h = torch.sigmoid(o) * torch.tanh(c)
        return self.head(h).view(B, self.P, self.C), h


def normalize(X: np.ndarray, how: str) -> np.ndarray:
    if how == "z":
        sd = X.std(0)
        return (X - X.mean(0)) / np.where(sd > 0, sd, 1)
    if how == "minmax":
        lo, hi = X.min(0), X.max(0)
        return (X - lo) / np.where(hi > lo, hi - lo, 1)
    return X


def windows(Xt, idx, L, P):
    ar = torch.arange(L + P, device=Xt.device)
    w = Xt[idx[:, None] + ar]
    return w[:, :L], w[:, L:]


def _per_sample_sqgrad(model, Xt, M, L, P, bs=4096):
    """||grad_theta l(x)||^2 de cada ventana sobre TODOS los parámetros."""
    params = {k: v.detach() for k, v in model.named_parameters()}

    def loss_fn(p, xb, yb):
        pred, _ = functional_call(model, p, (xb[None],))
        return ((pred[0] - yb) ** 2).mean()

    g_fn = vmap(grad(loss_fn), in_dims=(None, 0, 0))
    out = []
    for s in range(0, M, bs):
        idx = torch.arange(s, min(s + bs, M), device=Xt.device)
        xb, yb = windows(Xt, idx, L, P)
        g = g_fn(params, xb, yb)
        out.append(sum((v ** 2).flatten(1).sum(1) for v in g.values()))
    return torch.cat(out)


@torch.no_grad()
def _stats(model, Xt, M, L, P, bs=8192):
    loss, hsq = [], []
    for s in range(0, M, bs):
        idx = torch.arange(s, min(s + bs, M), device=Xt.device)
        xb, yb = windows(Xt, idx, L, P)
        pred, h = model(xb)
        loss.append(((pred - yb) ** 2).mean(dim=(1, 2)))
        hsq.append((h ** 2).sum(1))
    return torch.cat(loss), torch.cat(hsq)


def run(X: np.ndarray, L: int, P: int, seed: int, cfg: Cfg, device: str = "cpu",
        weights: np.ndarray | None = None, keep_g2: bool = False) -> dict:
    """Entrena y devuelve scores por ventana: self-influence (según cfg),
    pérdida final y pérdida por época (para diagnósticos). Con keep_g2 devuelve
    también la contribución de cada época, η_k‖∇ℓ‖² (épocas × ventanas)."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    Xn = normalize(X.astype(np.float64), cfg.norm)
    n, C = Xn.shape
    M = n - L - P + 1
    Xt = torch.as_tensor(Xn, dtype=torch.float32, device=device)
    model = LSTM(C, P, cfg.hidden).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.9, patience=3) if cfg.sched else None
    n_val = int(M * cfg.val_frac)
    tr_idx = torch.arange(M - n_val, device=device)
    va_idx = torch.arange(M - n_val, M, device=device)
    w_all = torch.ones(M, device=device) if weights is None else torch.as_tensor(weights, dtype=torch.float32, device=device)
    gen = torch.Generator().manual_seed(seed)
    D = P * C
    si = torch.zeros(M, device=device)
    lrs, losses, g2s = [], [], []
    best, bad = math.inf, 0
    for _ in range(cfg.max_epochs):
        lr_now = opt.param_groups[0]["lr"]
        perm = tr_idx[torch.randperm(len(tr_idx), generator=gen).to(device)]
        model.train()
        for b in perm.split(cfg.batch):
            xb, yb = windows(Xt, b, L, P)
            pred, _ = model(xb)
            l = ((pred - yb) ** 2).mean(dim=(1, 2))
            wb = w_all[b]
            loss = (l * wb).sum() / wb.sum().clamp_min(1e-12)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        model.eval()
        l_all, h_all = _stats(model, Xt, M, L, P)
        if cfg.grad == "last":
            g2 = (4.0 / D) * l_all * (1 + h_all)
        else:
            g2 = _per_sample_sqgrad(model, Xt, M, L, P)
        si += (lr_now if cfg.lr_weight else 1.0) * g2
        if keep_g2:
            g2s.append(((lr_now if cfg.lr_weight else 1.0) * g2).cpu().numpy().astype(np.float32))
        lrs.append(lr_now)
        losses.append(l_all.cpu().numpy().astype(np.float32))
        ref = va_idx if n_val > 0 else tr_idx
        mon = float((l_all[ref] * w_all[ref]).sum() / w_all[ref].sum())
        if sched is not None:
            sched.step(mon)
        if mon < best * (1 - 1e-4):
            best, bad = mon, 0
        else:
            bad += 1
            if bad >= cfg.patience:
                break
    out = {"si": si.cpu().numpy().astype(np.float64), "loss_final": losses[-1].astype(np.float64),
           "loss": np.stack(losses), "epochs": len(lrs)}
    if keep_g2:
        out["g2"] = np.stack(g2s)
    return out

## 1. El método original

1. Se toma la serie, sin etiquetas.
2. Se entrena un LSTM para que prediga la serie.
3. Con TracIn se mide cuánto le costó aprender cada ventana (4 instantes de entrada + 1 que predice).
4. El puntaje de cada ventana se pasa a los instantes: el artículo suma, en cada instante, el puntaje de todas las ventanas
   que lo contienen.
5. Lo que supera un umbral se marca como anomalía. El artículo elige ese umbral con etiquetas.

Ejemplo en una serie sintética con anomalías puntuales (tipo 0). Las franjas rojas son las anomalías reales.

In [ ]:
X1, y1 = make_dataset("0", seed=0)
r1 = run(X1, 4, 1, seed=0, cfg=Cfg())
n1, m1 = len(y1), valid_mask(len(y1))
s_original = project_sum(r1["si"], 5, n1)
print(f"Épocas: {r1['epochs']} · AUPRC del método original: {average_precision_score(y1[m1], s_original[m1]):.3f}")

fig, ax = plt.subplots(2, 1, figsize=(11, 4), sharex=True)
ax[0].plot(X1[:, 0], lw=.8, color="k"); ax[0].set_ylabel("serie (canal 0)")
ax[1].plot(s_original / s_original.max(), lw=.9, color="k"); ax[1].set_ylabel("puntaje TracIn")
for a in ax:
    for i in np.flatnonzero(y1):
        a.axvspan(i - .5, i + .5, color="tab:red", alpha=.25, lw=0)
ax[1].set_xlabel("instante"); plt.tight_layout(); plt.show()

Los puntajes altos coinciden con las anomalías: el modelo sí tuvo más dificultad con ellas. Pero cada pico se ve como una
meseta de varios instantes; eso es lo que corrige la mejora 3.

## 2. Correr los experimentos

Aquí se hace todo el cálculo; las secciones siguientes sólo analizan los resultados.

**Experimento A, 15 sintéticos.** Los 15 conjuntos de Lai et al. (400 instantes, 5 canales, anomalías puntuales y de
patrón, solas o combinadas), 5 semillas (nosotros: 20). Ventana fija de 4 + 1. Para cada serie se entrena el LSTM
una vez y se reentrena dos veces para la mejora 2. Se calculan: TracIn original, sólo mejora 2, sólo mejora 3, mejoras 2 + 3,
el error de predicción final del mismo LSTM y LOF.

**Experimento B, contaminación.** Una serie de 4000 instantes con los cinco tipos de anomalía, con proporción nominal de
anomalías [0.01, 0.15, 0.3] (real ≈ 1 %, 14 % y 26 %), 3 semillas (nosotros: 7 niveles × 10 semillas). Además de lo anterior
se calcula el recorte adaptativo (mejoras 1 + 2), que se reentrena tres veces. Aquí se mide con AUROC, que no depende de la
proporción de anomalías.

In [ ]:
SEMILLAS_A = 5   # nosotros: 20

def evaluar(y, s):
    """AUPRC, F1 con el umbral del artículo y F1 con los umbrales sin etiquetas (misma segunda mitad)."""
    s = np.asarray(s, dtype=np.float32).astype(np.float64)   # igual que en nuestros archivos
    e = evaluate(y, s, 0.85)
    k = len(s) // 2
    for nombre, thr in unsupervised_thresholds(s).items():
        e[f"U_{nombre}_bin"], e[f"U_{nombre}_mac"] = f1_pair(y[k:].astype(bool), s[k:] >= thr)
    return e

filas, t0 = [], time.time()
for i, ds in enumerate(DATASETS):
    for seed in range(SEMILLAS_A):
        X, y = make_dataset(ds, seed)
        n = len(y); m = valid_mask(n)
        r = run(X, 4, 1, seed, Cfg())
        s2 = r["si"]
        for _ in range(2):                                   # mejora 2: recorte z > 2, 2 rondas
            s2 = run(X, 4, 1, seed, Cfg(), weights=weights_from_influence(s2, "trim"))["si"]
        Xz = normalize(X.astype(np.float64), "z").astype(np.float32)
        scores = {"tracin": project_sum(r["si"], 5, n), "m2": project_sum(s2, 5, n),
                  "m3": project_targets_mean(r["si"], 4, 1, n), "m23": project_targets_mean(s2, 4, 1, n),
                  "pd": project_targets_mean(r["loss_final"], 4, 1, n), "lof": lof(Xz, 1)}
        for nombre, sc in scores.items():
            filas.append(dict(ds=ds, seed=seed, m=nombre, **evaluar(y[m], sc[m])))
    print(f"A: {i + 1}/15 conjuntos · {time.time() - t0:.0f} s")
A = pd.DataFrame(filas)

In [ ]:
NIVELES = [0.01, 0.15, 0.3]
SEMILLAS_B = 3   # nosotros: 10

filas, filas_thr, t0 = [], [], time.time()
for r_ in NIVELES:
    for seed in range(SEMILLAS_B):
        X, y = make_dataset("01234", 10_000 + seed, length=4000, ratio=r_ / 5)
        n = len(y); m = valid_mask(n)
        base = run(X, 4, 1, seed, Cfg())
        s2 = base["si"]
        for _ in range(2):                                   # mejora 2: recorte z > 2, 2 rondas
            s2 = run(X, 4, 1, seed, Cfg(), weights=weights_from_influence(s2, "trim"))["si"]
        s12 = base["si"]
        for _ in range(3):                                   # mejoras 1 + 2: recorte con el umbral IQR, 3 rondas
            s12 = run(X, 4, 1, seed, Cfg(), weights=adapt_weights(s12))["si"]
        Xz = normalize(X.astype(np.float64), "z").astype(np.float32)
        scores = {"TracIn original": project_sum(base["si"], 5, n), "Sólo mejora 2": project_sum(s2, 5, n),
                  "Sólo mejora 3": project_targets_mean(base["si"], 4, 1, n), "Mejoras 1 + 2": project_sum(s12, 5, n),
                  "Mejoras 2 + 3": project_targets_mean(s2, 4, 1, n), "Mejoras 1 + 2 + 3": project_targets_mean(s12, 4, 1, n),
                  "LOF": lof(Xz, 1), "Error LSTM": project_targets_mean(base["loss_final"], 4, 1, n)}
        fila = dict(nivel=r_, seed=seed, anomalias=y[m].mean())
        for k, v in scores.items():
            fila[k] = roc_auc_score(y[m], np.asarray(v, dtype=np.float32)[m])
        filas.append(fila)
        for k, nombre in (("TracIn original", "TracIn original"), ("Sólo mejora 3", "TracIn con mejora 3")):
            e = evaluar(y[m], scores[k][m])
            filas_thr.append(dict(nivel=r_, seed=seed, score=nombre, dyn=e["D_mac"], iqr=e["U_iqr_mac"], mad=e["U_mad_mac"]))
        print(f"B: nivel {r_:.2f} · semilla {seed} · {time.time() - t0:.0f} s")
B = pd.DataFrame(filas); Bthr = pd.DataFrame(filas_thr)

In [ ]:
# Nuestras cifras de referencia (corridas completas) y utilidades para las tablas
REF_A_MISMAS = {"lof": {"auprc": 0.6429, "D_mac": 0.7748, "U_iqr_mac": 0.7856, "U_mad_mac": 0.7883, "auprc_puntual": 0.7356, "auprc_patron": 0.5415}, "m2": {"auprc": 0.6133, "D_mac": 0.6984, "U_iqr_mac": 0.722, "U_mad_mac": 0.7246, "auprc_puntual": 0.2815, "auprc_patron": 0.7731}, "m23": {"auprc": 0.7928, "D_mac": 0.8345, "U_iqr_mac": 0.8408, "U_mad_mac": 0.8396, "auprc_puntual": 0.8516, "auprc_patron": 0.6857}, "m3": {"auprc": 0.7807, "D_mac": 0.8255, "U_iqr_mac": 0.8334, "U_mad_mac": 0.8294, "auprc_puntual": 0.8481, "auprc_patron": 0.6688}, "pd": {"auprc": 0.6977, "D_mac": 0.793, "U_iqr_mac": 0.7976, "U_mad_mac": 0.8016, "auprc_puntual": 0.8479, "auprc_patron": 0.5307}, "tracin": {"auprc": 0.5834, "D_mac": 0.7031, "U_iqr_mac": 0.7157, "U_mad_mac": 0.7259, "auprc_puntual": 0.279, "auprc_patron": 0.7107}}
REF_A_20 = {"lof": {"auprc": 0.6536, "D_mac": 0.7801, "U_iqr_mac": 0.7826, "U_mad_mac": 0.787, "auprc_puntual": 0.7657, "auprc_patron": 0.537}, "m2": {"auprc": 0.5964, "D_mac": 0.696, "U_iqr_mac": 0.7124, "U_mad_mac": 0.7227, "auprc_puntual": 0.2842, "auprc_patron": 0.7869}, "m23": {"auprc": 0.7878, "D_mac": 0.8247, "U_iqr_mac": 0.8253, "U_mad_mac": 0.8257, "auprc_puntual": 0.8505, "auprc_patron": 0.693}, "m3": {"auprc": 0.782, "D_mac": 0.8165, "U_iqr_mac": 0.8154, "U_mad_mac": 0.8147, "auprc_puntual": 0.8429, "auprc_patron": 0.6872}, "pd": {"auprc": 0.6978, "D_mac": 0.7872, "U_iqr_mac": 0.7876, "U_mad_mac": 0.788, "auprc_puntual": 0.8449, "auprc_patron": 0.5508}, "tracin": {"auprc": 0.5755, "D_mac": 0.6971, "U_iqr_mac": 0.7079, "U_mad_mac": 0.7208, "auprc_puntual": 0.2861, "auprc_patron": 0.7577}}
REF_B_MISMAS = {"0.01": {"TracIn original": 0.9565, "Sólo mejora 2": 0.9556, "Sólo mejora 3": 0.9446, "Mejoras 1 + 2": 0.9316, "Mejoras 2 + 3": 0.9426, "Mejoras 1 + 2 + 3": 0.9192, "LOF": 0.9, "Error LSTM": 0.8896}, "0.15": {"TracIn original": 0.8472, "Sólo mejora 2": 0.8741, "Sólo mejora 3": 0.889, "Mejoras 1 + 2": 0.8788, "Mejoras 2 + 3": 0.914, "Mejoras 1 + 2 + 3": 0.9198, "LOF": 0.8625, "Error LSTM": 0.8434}, "0.30": {"TracIn original": 0.7868, "Sólo mejora 2": 0.8122, "Sólo mejora 3": 0.871, "Mejoras 1 + 2": 0.8191, "Mejoras 2 + 3": 0.895, "Mejoras 1 + 2 + 3": 0.9015, "LOF": 0.8633, "Error LSTM": 0.8219}}
REF_B_UMBRAL = {"TracIn original": {"0.01": {"dyn": 0.6787, "iqr": 0.6755, "mad": 0.684}, "0.15": {"dyn": 0.6958, "iqr": 0.6945, "mad": 0.6967}, "0.30": {"dyn": 0.6763, "iqr": 0.5609, "mad": 0.6727}}, "TracIn con mejora 3": {"0.01": {"dyn": 0.8793, "iqr": 0.6726, "mad": 0.6721}, "0.15": {"dyn": 0.8283, "iqr": 0.8236, "mad": 0.8269}, "0.30": {"dyn": 0.8244, "iqr": 0.8084, "mad": 0.8156}}}
REF_B_COMPLETO = {"rate": [0.0115, 0.049, 0.0952, 0.138, 0.1819, 0.2227, 0.2585], "TracIn original": [0.9536, 0.9311, 0.8924, 0.8582, 0.8321, 0.8113, 0.789], "Sólo mejora 2": [0.9537, 0.9352, 0.911, 0.8807, 0.8559, 0.8346, 0.813], "Sólo mejora 3": [0.9452, 0.9413, 0.9194, 0.9015, 0.8966, 0.8792, 0.8732], "Mejoras 1 + 2": [0.9461, 0.9347, 0.913, 0.8857, 0.8624, 0.8411, 0.8214], "Mejoras 2 + 3": [0.9444, 0.9453, 0.941, 0.9252, 0.9207, 0.9037, 0.8962], "Mejoras 1 + 2 + 3": [0.9369, 0.9441, 0.9429, 0.9313, 0.9295, 0.9118, 0.9048], "LOF": [0.9105, 0.8978, 0.9002, 0.8813, 0.8684, 0.8727, 0.8521], "Error LSTM": [0.8819, 0.8869, 0.8591, 0.8517, 0.841, 0.8306, 0.8226], "IForest": [0.8069, 0.7725, 0.7855, 0.7851, 0.7795, 0.7805, 0.7664], "OCSVM": [0.632, 0.6637, 0.6737, 0.6808, 0.6842, 0.6835, 0.6748], "Autoencoder": [0.671, 0.6438, 0.6598, 0.6427, 0.6596, 0.6539, 0.6393], "Matrix Profile": [0.854, 0.8455, 0.8053, 0.7727, 0.7407, 0.721, 0.7111]}
NOMBRES_A = {"tracin": "TracIn original", "m2": "Sólo mejora 2", "m3": "Sólo mejora 3", "m23": "Mejoras 2 + 3",
             "pd": "Error de predicción del LSTM", "lof": "LOF"}
resA = A.groupby("m")[["auprc", "D_mac", "U_iqr_mac", "U_mad_mac"]].mean()
resA["auprc_puntual"] = A[A.ds.isin(["0", "1"])].groupby("m").auprc.mean()
resA["auprc_patron"] = A[A.ds.isin(["2", "3", "4"])].groupby("m").auprc.mean()
resB = B.groupby("nivel").mean(numeric_only=True)
ETIQ_B = [f"{v * 100:.0f} %" for v in resB.anomalias]

def tabla_A(claves, col="auprc"):
    """Métrica del experimento A: este notebook vs nuestra corrida (mismas semillas y 20 semillas)."""
    return pd.DataFrame({"Este notebook": [resA.loc[k, col] for k in claves],
                         "Nuestra corrida (mismas semillas)": [REF_A_MISMAS[k][col] for k in claves],
                         "Nuestra corrida (20 semillas)": [REF_A_20[k][col] for k in claves]},
                        index=[NOMBRES_A[k] for k in claves]).round(3)

def tabla_B(metodos):
    """AUROC del experimento B por nivel de anomalías, con nuestra referencia de las mismas semillas."""
    t = resB[metodos].T; t.columns = [f"{e} (notebook)" for e in ETIQ_B]
    ref = pd.DataFrame(REF_B_MISMAS).loc[metodos]; ref.columns = [f"{e} (nuestra)" for e in ETIQ_B]
    return pd.concat([t, ref], axis=1)[[c for p in zip(t.columns, ref.columns) for c in p]].round(3)

def grafica_B(metodos, titulo):
    """Curvas de nuestra corrida completa (10 semillas, 7 niveles) y puntos de este notebook."""
    x = [v * 100 for v in REF_B_COMPLETO["rate"]]
    plt.figure(figsize=(7, 3.4))
    for k in metodos:
        l, = plt.plot(x, REF_B_COMPLETO[k], "--" if k in ("LOF", "Error LSTM") else "-", lw=1.5, label=k)
        if k in resB:
            plt.plot(resB.anomalias * 100, resB[k], "o", color=l.get_color(), ms=5)
    plt.title(titulo + "\nlíneas: nuestra corrida completa · puntos: este notebook", fontsize=9, loc="left")
    plt.xlabel("% de instantes anómalos"); plt.ylabel("AUROC"); plt.legend(frameon=False, fontsize=8)
    plt.tight_layout(); plt.show()

print("Listo. Las secciones siguientes analizan estos resultados.")

## 3. Mejora 1: umbral sin etiquetas

**Qué es.** El artículo elige el umbral del paso 5 con etiquetas: prueba varios umbrales en la primera mitad de la serie,
se queda con el que da mejor F1 y lo aplica en la segunda. La mejora lo sustituye por reglas estadísticas sobre los propios
puntajes, sin etiquetas:
* **IQR:** se marca lo que supera Q3 + 1.5 · (Q3 − Q1).
* **MAD:** se marca lo que supera la mediana + 3 desviaciones absolutas medianas.

**Por qué.** En un problema no supervisado no hay etiquetas; un método que las necesita para decidir qué es anomalía no se
puede usar en la práctica. El objetivo no es detectar mejor, sino no depender de etiquetas.

**Cómo se mide.** Es un umbral: no cambia el orden de los puntajes, así que no cambia el AUPRC. Se mide con F1 (macro,
como el artículo en sintéticos) en la misma segunda mitad de la serie.

In [ ]:
print("Sintéticos: F1 macro con el umbral del artículo (con etiquetas) y con IQR y MAD (sin etiquetas)")
t = resA.loc[["tracin", "m3"], ["D_mac", "U_iqr_mac", "U_mad_mac"]].copy()
t.index = ["TracIn original", "TracIn con mejora 3"]
t.columns = ["Con etiquetas", "IQR", "MAD"]
t["Con etiquetas (nuestra)"] = [REF_A_MISMAS[k]["D_mac"] for k in ("tracin", "m3")]
t["IQR (nuestra)"] = [REF_A_MISMAS[k]["U_iqr_mac"] for k in ("tracin", "m3")]
t["MAD (nuestra)"] = [REF_A_MISMAS[k]["U_mad_mac"] for k in ("tracin", "m3")]
display(t.round(3))

print("Serie contaminada: F1 macro con y sin etiquetas")
t = Bthr.groupby(["score", "nivel"])[["dyn", "iqr", "mad"]].mean().unstack("nivel").round(3)
t.columns = [f"{ {'dyn': 'Con etiquetas', 'iqr': 'IQR', 'mad': 'MAD'}[a] } · {e}" for a in ("dyn", "iqr", "mad") for e in ETIQ_B]
display(t)
print("Nuestra corrida, mismas semillas")
display(pd.DataFrame({k: [v[r]["dyn"] for r in v] + [v[r]["iqr"] for r in v] + [v[r]["mad"] for r in v]
                      for k, v in REF_B_UMBRAL.items()}, index=t.columns).T.loc[t.index].round(3))

**Resultado.**
* En los sintéticos, IQR y MAD dan prácticamente el mismo F1 que el umbral con etiquetas: el método queda completamente no
  supervisado sin perder desempeño.
* En la serie contaminada se ve el límite: la regla adecuada depende de cuántas anomalías hay. Con la mejora 3, IQR falla
  cuando las anomalías son muy pocas (1 %), porque marca demasiados instantes, e iguala al umbral con etiquetas desde 5 %.
  Con el TracIn original, IQR falla con mucha contaminación y MAD aguanta.
* En datos reales (sección 8) pasa lo mismo: cada conjunto prefiere una regla distinta.

**¿Sirvió?** Sí, con un matiz: elimina las etiquetas sin costo en sintéticos, pero falta una forma de elegir la regla según
la proporción de anomalías.

## 4. Mejora 2: reponderación

**Qué es.** Se calcula la self-influence, se pone peso cero a las ventanas más sospechosas (z robusto mayor que 2), se
reentrena el LSTM desde cero con esos pesos y se repite. El puntaje final es la self-influence del último entrenamiento.

**Por qué.** TracIn supone que las anomalías son raras y por eso difíciles de aprender. Si la serie tiene muchas, el modelo
las ve con frecuencia, las aprende y su error baja: TracIn deja de distinguirlas. Al quitarles peso, el modelo aprende sobre
todo lo normal y las anomalías le siguen costando. Es la idea del aprendizaje robusto con datos ruidosos (Shen y Sanghavi,
2019). Es la única mejora que cambia el entrenamiento.

**Variante con la mejora 1 (mejoras 1 + 2).** En lugar de z > 2, se quitan las ventanas que superan el umbral IQR; así se
recorta más cuanto más contaminada está la serie.

In [ ]:
print("Sintéticos (AUPRC): la mejora 2 sola")
display(tabla_A(["tracin", "m2"]))

print("Serie contaminada (AUROC): la mejora 2 sola y con el criterio de la mejora 1")
display(tabla_B(["TracIn original", "Sólo mejora 2", "Mejoras 1 + 2"]))
grafica_B(["TracIn original", "Sólo mejora 2", "Mejoras 1 + 2", "LOF", "Error LSTM"], "Mejora 2")

**Resultado.**
* En los sintéticos de 400 instantes casi no cambia: hay pocas anomalías y el modelo no alcanza a aprenderlas, así que no
  hay nada que corregir.
* En la serie contaminada, el TracIn original se degrada mucho al aumentar las anomalías. La mejora 2 sola recupera una parte
  pequeña de esa caída, de forma consistente; con el criterio adaptativo de la mejora 1, un poco más.
* No basta sola: con 26 % de anomalías sigue por debajo de LOF y del error de predicción. El recorte sólo logra quitar cerca
  de un tercio de las ventanas anómalas, porque el modelo ya aprendió las demás.
* En datos reales (sección 8) el efecto es mixto: ayuda en GECCO, empeora Credit Card y no cambia los otros dos.

**¿Sirvió?** En parte: ataca el problema correcto, pero sola aporta poco. Funciona bien combinada con la mejora 3
(sección 6).

## 5. Mejora 3: proyección al objetivo

**Qué es.** Cambia el paso 4: el puntaje de cada ventana se asigna sólo al instante que la ventana predice, no a todos sus
instantes. Cada instante recibe el promedio de las ventanas que lo predicen. El modelo y el entrenamiento no cambian.

**Por qué.** El error de una ventana mide qué tan mal predijo su salida, así que la anomalía está en lo que no pudo
predecir, no en lo que vio. Si en t5 hay un pico, la ventana que predice t5 falla; el artículo reparte ese puntaje también a
t1–t4, que son normales, y el pico vuelve a sumar en las ventanas siguientes que lo tienen como entrada. Un pico de un
instante se vuelve una meseta. Es la forma en que los detectores de error de predicción asignan el error (Malhotra et al.,
2015).

La misma serie de la sección 1, con el mismo modelo y la misma self-influence:

In [ ]:
s_mejora3 = project_targets_mean(r1["si"], 4, 1, n1)
fig, ax = plt.subplots(2, 1, figsize=(11, 4), sharex=True)
for a, sc, t in [(ax[0], s_original, "Método original"), (ax[1], s_mejora3, "Mejora 3")]:
    a.plot(sc / sc.max(), lw=.9, color="k")
    a.set_title(f"{t} · AUPRC {average_precision_score(y1[m1], sc[m1]):.3f}", loc="left", fontsize=10)
    for i in np.flatnonzero(y1):
        a.axvspan(i - .5, i + .5, color="tab:red", alpha=.25, lw=0)
ax[1].set_xlabel("instante (rojo: anomalía)"); plt.tight_layout(); plt.show()

print("Sintéticos (AUPRC), total y por tipo de anomalía")
t = tabla_A(["tracin", "m3"])
t["Puntuales, tipos 0 y 1 (notebook)"] = [resA.loc[k, "auprc_puntual"] for k in ("tracin", "m3")]
t["Puntuales (nuestra, mismas semillas)"] = [REF_A_MISMAS[k]["auprc_puntual"] for k in ("tracin", "m3")]
t["De patrón, tipos 2-4 (notebook)"] = [resA.loc[k, "auprc_patron"] for k in ("tracin", "m3")]
t["De patrón (nuestra, mismas semillas)"] = [REF_A_MISMAS[k]["auprc_patron"] for k in ("tracin", "m3")]
display(t.round(3))

print("Serie contaminada (AUROC)")
display(tabla_B(["TracIn original", "Sólo mejora 3"]))
grafica_B(["TracIn original", "Sólo mejora 3", "LOF", "Error LSTM"], "Mejora 3")

**Resultado.**
* Con la mejora 3 el puntaje se queda en el instante de la anomalía en lugar de formar mesetas.
* Es la mejora con más efecto: el AUPRC en sintéticos sube de forma muy marcada, sobre todo en anomalías puntuales, que eran
  la debilidad del método original.
* En la serie contaminada es la que más frena la degradación por sí sola.
* En datos reales (sección 8) es la mejor en Credit Card (fraudes puntuales). En GECCO, con anomalías largas, la proyección
  original funciona un poco mejor.

**¿Sirvió?** Sí, y es la de mayor efecto.

## 6. Las mejoras combinadas

* **Mejoras 2 + 3:** reponderación (z > 2) con la proyección al objetivo.
* **Mejoras 1 + 2 + 3:** reponderación con el umbral IQR como criterio de recorte y proyección al objetivo.

In [ ]:
print("Sintéticos (AUPRC)")
display(tabla_A(["tracin", "m2", "m3", "m23"]))

print("Serie contaminada (AUROC)")
display(tabla_B(["TracIn original", "Sólo mejora 2", "Sólo mejora 3", "Mejoras 1 + 2", "Mejoras 2 + 3", "Mejoras 1 + 2 + 3"]))
print("Caída de 1 % a 26 % de anomalías (nuestra corrida completa, 10 semillas)")
display(pd.Series({k: REF_B_COMPLETO[k][-1] - REF_B_COMPLETO[k][0] for k in
                   ["TracIn original", "Sólo mejora 2", "Mejoras 1 + 2", "Sólo mejora 3", "Mejoras 2 + 3", "Mejoras 1 + 2 + 3",
                    "LOF", "Error LSTM"]}).round(3).to_frame("caída de AUROC"))
grafica_B(["TracIn original", "Sólo mejora 3", "Mejoras 2 + 3", "Mejoras 1 + 2 + 3", "LOF", "Error LSTM"], "Mejoras combinadas")

**Resultado.** Las mejoras se complementan. En los sintéticos manda la mejora 3. Con contaminación, las tres juntas son las
que menos se degradan y el mejor detector cuando hay muchas anomalías; el costo es una ligera pérdida con 1 % y reentrenar
el modelo tres veces más.

## 7. Comparación con otros métodos de detección

Todos los métodos reciben la misma serie, dan un puntaje a cada instante y se califican igual contra las anomalías reales.
Hay dos grupos:
* **Usan el mismo LSTM:** TracIn original, TracIn con nuestras mejoras y el error de predicción final. Sólo cambia cómo se
  aprovecha el modelo; es la comparación más limpia.
* **No usan el LSTM:** LOF (densidad de vecinos) en este notebook; en nuestras corridas completas también Isolation Forest,
  One-Class SVM, un autoencoder y Matrix Profile.

In [ ]:
print("Sintéticos (AUPRC)")
display(tabla_A(["m3", "pd", "lof", "tracin"]))

print("Serie contaminada (AUROC), nuestra corrida completa con todos los métodos")
t = pd.DataFrame({k: REF_B_COMPLETO[k] for k in ["Mejoras 1 + 2 + 3", "Sólo mejora 3", "LOF", "Error LSTM", "IForest",
                                                  "Matrix Profile", "OCSVM", "Autoencoder", "TracIn original"]}).T
t.columns = [f"{v * 100:.0f} %" for v in REF_B_COMPLETO["rate"]]
display(t.round(3))

**Resultado.** Con la misma métrica, el TracIn original queda por debajo del error de predicción y de LOF en sintéticos. Con
la mejora 3 queda primero, y con las tres mejoras es el mejor en todos los niveles de contaminación. La comparación con el
error de predicción del mismo modelo muestra que usar el proceso de aprendizaje sí aporta, una vez que el puntaje se asigna
bien.

## 8. Resultados completos y resumen

Estas tablas no se calculan aquí: son de nuestras corridas completas (sintéticos con 20 semillas y la mejor de 9
configuraciones de ventana; reales con 3 semillas). Los datos reales (Credit Card, CICIDS, GECCO y SWAN-SF) pesan cerca de
600 MB y tardan horas, por eso no se corren en este notebook.

In [ ]:
FULL = {"sinteticos": {"TracIn (artículo)": {"AUPRC": 0.6063, "F1 macro": 0.7029}, "TracIn proy. objetivo": {"AUPRC": 0.8323, "F1 macro": 0.8318}, "Error LSTM (PD)": {"AUPRC": 0.8097, "F1 macro": 0.8316}, "LOF": {"AUPRC": 0.7363, "F1 macro": 0.7966}, "One-Class SVM": {"AUPRC": 0.6081, "F1 macro": 0.7442}, "Isolation Forest": {"AUPRC": 0.6639, "F1 macro": 0.7357}, "Autoencoder": {"AUPRC": 0.66, "F1 macro": 0.7107}, "Matrix Profile": {"AUPRC": 0.5533, "F1 macro": 0.6862}}, "reales_auprc": {"creditcard": {"IForest": 0.1603, "OCSVM": 0.1315, "LOF": 0.0144, "Autoencoder": 0.1212, "Matrix Profile": 0.0015, "TracIn": 0.1159, "TracIn proy. objetivo": 0.2142}, "cicids": {"IForest": 0.0222, "OCSVM": 0.0174, "LOF": 0.0182, "Autoencoder": 0.0212, "Matrix Profile": 0.0097, "TracIn": 0.0169, "TracIn proy. objetivo": 0.0153}, "gecco": {"IForest": 0.4139, "OCSVM": 0.4633, "LOF": 0.0861, "Autoencoder": 0.6212, "Matrix Profile": 0.0906, "TracIn": 0.4607, "TracIn proy. objetivo": 0.4375}, "swan_sf": {"IForest": 0.6161, "OCSVM": 0.4638, "LOF": 0.4191, "Autoencoder": 0.4133, "Matrix Profile": 0.3272, "TracIn": 0.3393, "TracIn proy. objetivo": 0.3764}}, "reales_mejoras": {"creditcard": {"tracin": 0.1159, "tracin_tgt": 0.2142, "rw_soft_r1": 0.0852, "rw_trim_r1": 0.0797, "rw_trim_r1_tgt": 0.2012}, "cicids": {"tracin": 0.0169, "tracin_tgt": 0.0153, "rw_soft_r1": 0.0174, "rw_trim_r1": 0.0169, "rw_trim_r1_tgt": 0.0159}, "gecco": {"tracin": 0.4607, "tracin_tgt": 0.4375, "rw_soft_r1": 0.5222, "rw_trim_r1": 0.5108, "rw_trim_r1_tgt": 0.4986}, "swan_sf": {"tracin": 0.3393, "tracin_tgt": 0.3764, "rw_soft_r1": 0.3405, "rw_trim_r1": 0.347, "rw_trim_r1_tgt": 0.3718}}, "reales_umbral": {"TracIn (artículo)": {"creditcard": {"D_bin": 0.2255, "S_pct": 0.0203, "U_iqr": 0.018, "U_mad": 0.0165, "U_log_iqr": 0.0259, "U_log_mad": 0.0249, "U_pot": 0.237}, "cicids": {"D_bin": 0.0026, "S_pct": 0.0284, "U_iqr": 0.0021, "U_mad": 0.0025, "U_log_iqr": 0.0, "U_log_mad": 0.0, "U_pot": 0.0}, "gecco": {"D_bin": 0.3255, "S_pct": 0.132, "U_iqr": 0.1836, "U_mad": 0.1378, "U_log_iqr": 0.3252, "U_log_mad": 0.3325, "U_pot": 0.1052}, "swan_sf": {"D_bin": 0.378, "S_pct": 0.2862, "U_iqr": 0.2724, "U_mad": 0.2908, "U_log_iqr": 0.153, "U_log_mad": 0.1378, "U_pot": 0.009}}, "TracIn proy. objetivo": {"creditcard": {"D_bin": 0.2824, "S_pct": 0.0237, "U_iqr": 0.0271, "U_mad": 0.0248, "U_log_iqr": 0.0593, "U_log_mad": 0.0687, "U_pot": 0.3135}, "cicids": {"D_bin": 0.0029, "S_pct": 0.0133, "U_iqr": 0.002, "U_mad": 0.0023, "U_log_iqr": 0.0032, "U_log_mad": 0.0019, "U_pot": 0.0}, "gecco": {"D_bin": 0.3181, "S_pct": 0.1183, "U_iqr": 0.0748, "U_mad": 0.0667, "U_log_iqr": 0.2952, "U_log_mad": 0.3077, "U_pot": 0.1032}, "swan_sf": {"D_bin": 0.3826, "S_pct": 0.3255, "U_iqr": 0.2928, "U_mad": 0.3317, "U_log_iqr": 0.1026, "U_log_mad": 0.084, "U_pot": 0.0084}}}}
print("Sintéticos, todos los métodos (mejor configuración de cada uno)")
display(pd.DataFrame(FULL["sinteticos"]).T.rename(index={"TracIn (artículo)": "TracIn original",
        "TracIn proy. objetivo": "TracIn con mejora 3", "Error LSTM (PD)": "Error de predicción"}).round(3))
print("Reales: AUPRC de cada método")
display(pd.DataFrame(FULL["reales_auprc"]).rename(index={"TracIn": "TracIn original",
        "TracIn proy. objetivo": "TracIn con mejora 3"}).round(3))
print("Reales: AUPRC con las mejoras 2 y 3 (reponderación de 1 ronda, criterio z > 2)")
display(pd.DataFrame(FULL["reales_mejoras"]).rename(index={"tracin": "TracIn original", "tracin_tgt": "Sólo mejora 3",
        "rw_soft_r1": "Sólo mejora 2 (suave)", "rw_trim_r1": "Sólo mejora 2 (recorte)", "rw_trim_r1_tgt": "Mejoras 2 + 3"}).round(3))
print("Reales: mejora 1, F1 binario con umbral con etiquetas (D_bin) y sin etiquetas (U_*), ventana 4 + 1")
for sc, d in FULL["reales_umbral"].items():
    print(sc); display(pd.DataFrame(d).round(3))

| Mejora | Qué cambia | ¿Sirvió? |
|---|---|---|
| 1 · Umbral sin etiquetas | El umbral final | Sí en sintéticos: iguala al umbral con etiquetas. La regla adecuada depende de la proporción de anomalías. |
| 2 · Reponderación | El entrenamiento | En parte: sola aporta poco; combinada con 1 y 3 deja la menor degradación con contaminación. En reales es mixta. |
| 3 · Proyección al objetivo | Cómo se asigna el puntaje | Sí: la de mayor efecto; mejor detector en sintéticos y en Credit Card. |